# SDOCX Format — Container Structure

Samsung Notes stores handwritten notes as `.sdocx` files. Despite the name suggesting
a document format, these are actually **ZIP archives** containing binary stroke data,
metadata, and media. The format is produced by the **Samsung S-Pen SDK**, used across
Galaxy Note and Galaxy Tab devices.

This notebook documents the container format by examining a real handwritten note.

In [32]:
from pathlib import Path

SAMPLE = Path("../samples/handwritten.sdocx")
SAMPLE = Path("../samples/Appunti vari_260629_230848.sdocx")
SAMPLE = Path("../samples/OnlyPensBlacksize10_260630_120309.sdocx")

In [ ]:
import sys
sys.path.insert(0, "..")

import zipfile
import struct
import hashlib
import datetime

from pysdocx import hexdump, dump_container, bg_color_from_note

## ZIP Contents

The `.sdocx` file is a standard ZIP archive. Let's list everything inside.

In [ ]:
print(dump_container(SAMPLE))

In [36]:
with zipfile.ZipFile(SAMPLE) as z:
    files = {info.filename: z.read(info.filename) for info in z.infolist()}

### File tree

```
handwritten.sdocx (ZIP)
├── pageIdInfo.dat          Page UUID with integrity hashes (140 B)
├── <uuid>.page             Stroke coordinates + attributes (~4.5 MB)
├── media/
│   ├── mediaInfo.dat       Media manifest with SHA-256 hash (131 B)
│   └── 0@page_0000000.spi  Page thumbnail, proprietary format (~454 KB)
├── note.note               Title, pen tools, background color (1.7 KB)
└── end_tag.bin             Timestamps, SDK marker (144 B)
```

**Conventions across all files:**
- All integers are **little-endian**
- Strings use **UTF-16LE** (in binary fields) or plain **ASCII** (UUIDs, hashes)
- Timestamps are **int64 milliseconds since Unix epoch** (UTC)
- Footer markers identify the SDK: `"Document for S-Pen SDK"`, `"Page for SAMSUNG S-Pen SDK"`, `"EOFX"`

---

## end_tag.bin

The document footer (144 bytes). Contains **creation and modification timestamps** as
int64 milliseconds since Unix epoch, and ends with the ASCII marker `"Document for S-Pen SDK"`.

| Offset | Size | Type | Content |
|--------|------|------|---------|
| `0x48` | 8 | i64 | Created timestamp (ms since epoch) |
| `0x50` | 8 | i64 | Modified timestamp (ms since epoch) |
| `0x7A` | 22 | ASCII | `"Document for S-Pen SDK"` |

In [ ]:
data = files["end_tag.bin"]
print(f"end_tag.bin — {len(data)} bytes\n")

ts_created = struct.unpack_from("<q", data, 0x48)[0]
ts_modified = struct.unpack_from("<q", data, 0x50)[0]
try:
    created = datetime.datetime.fromtimestamp(ts_created / 1000, tz=datetime.timezone.utc)
    modified = datetime.datetime.fromtimestamp(ts_modified / 1000, tz=datetime.timezone.utc)
    print(f"Created:   {created}")
    print(f"Modified:  {modified}")
except (ValueError, OverflowError, OSError):
    # Some files (e.g. ones without media) are a few bytes longer than the
    # 144-byte layout this offset assumes, shifting everything after the
    # header — not yet reverse-engineered. Show the raw values instead of
    # crashing the notebook.
    print(f"Created (raw i64):  {ts_created} (offset 0x48 didn't decode to a sane date)")
    print(f"Modified (raw i64): {ts_modified}")
print(f"Marker:    {data[-22:].decode('ascii')!r}")

In [38]:
print(hexdump(data))

  0000  92 00 a0 0f 00 00 00 00 8d 42 9c ae 75 55 06 00  ·········B··uU··
  0010  00 00 00 00 00 00 40 06 00 00 00 78 8e 45 00 00  ······@····x·E··
  0020  ff ff ff ff ff ff ff ff 00 00 a0 0f 00 00 74 97  ··············t·
  0030  33 a1 75 55 06 00 00 00 00 00 00 00 00 00 00 00  3·uU············
  0040  00 00 00 00 00 00 00 00 74 97 33 a1 75 55 06 00  ········t·3·uU··
  0050  74 97 33 a1 75 55 06 00 00 00 00 00 00 00 00 00  t·3·uU··········
  0060  00 00 02 00 00 00 02 00 00 00 ff ff ff ff ff ff  ················
  0070  ff ff 00 00 00 00 00 00 00 00 00 00 00 00 44 6f  ··············Do
  0080  63 75 6d 65 6e 74 20 66 6f 72 20 53 2d 50 65 6e  cument for S-Pen
  0090  20 53 44 4b                                       SDK


---

## pageIdInfo.dat

The page registry (140 bytes). Contains a **page UUID** as a length-prefixed UTF-16LE string,
sandwiched between two 32-byte values (likely integrity hashes or checksums).

| Offset | Size | Type | Content |
|--------|------|------|---------|
| `0x00` | 32 | bytes | Hash/checksum A |
| `0x20` | 2 | u16 | Page count |
| `0x22` | 2 | u16 | UUID string length (chars) |
| `0x24` | 72 | UTF-16LE | Page UUID (36 chars × 2 bytes) |
| `0x6C` | 32 | bytes | Hash/checksum B |

In [39]:
data = files["pageIdInfo.dat"]
print(f"pageIdInfo.dat \u2014 {len(data)} bytes\n")

hash_a = data[0x00:0x20]
count = struct.unpack_from("<H", data, 0x20)[0]
str_len = struct.unpack_from("<H", data, 0x22)[0]
uuid = data[0x24 : 0x24 + str_len * 2].decode("utf-16-le")
hash_b = data[0x6C:0x8C]

page_filename = [k for k in files if k.endswith(".page")][0]

print(f"Page count:    {count}")
print(f"Page UUID:     {uuid}")
print(f".page file:    {page_filename}")
print(f"UUID matches:  {uuid == page_filename.removesuffix('.page')}")
print(f"Hash A:        {hash_a.hex()}")
print(f"Hash B:        {hash_b.hex()}")

pageIdInfo.dat — 246 bytes

Page count:    2
Page UUID:     4302b93e-746a-11f1-9552-b7833a8a74d9
.page file:    4302b93e-746a-11f1-9552-b7833a8a74d9.page
UUID matches:  True
Hash A:        9e7cff7cdeb3fcc90176a986e51e00c399dfdbb90001f384b65cc02f7490cd30
Hash B:        62c805efa5d637e28f1743d45a257088c92a12fb66f343cff7e653cbcd6a3251


In [40]:
print(hexdump(data))

  0000  9e 7c ff 7c de b3 fc c9 01 76 a9 86 e5 1e 00 c3  ·|·|·····v······
  0010  99 df db b9 00 01 f3 84 b6 5c c0 2f 74 90 cd 30  ·········\·/t··0
  0020  02 00 24 00 34 00 33 00 30 00 32 00 62 00 39 00  ··$·4·3·0·2·b·9·
  0030  33 00 65 00 2d 00 37 00 34 00 36 00 61 00 2d 00  3·e·-·7·4·6·a·-·
  0040  31 00 31 00 66 00 31 00 2d 00 39 00 35 00 35 00  1·1·f·1·-·9·5·5·
  0050  32 00 2d 00 62 00 37 00 38 00 33 00 33 00 61 00  2·-·b·7·8·3·3·a·
  0060  38 00 61 00 37 00 34 00 64 00 39 00 62 c8 05 ef  8·a·7·4·d·9·b···
  0070  a5 d6 37 e2 8f 17 43 d4 5a 25 70 88 c9 2a 12 fb  ··7···C·Z%p··*··
  0080  66 f3 43 cf f7 e6 53 cb cd 6a 32 51 24 00 34 00  f·C···S··j2Q$·4·
  0090  33 00 30 00 33 00 35 00 37 00 31 00 38 00 2d 00  3·0·3·5·7·1·8·-·
  00a0  37 00 34 00 36 00 61 00 2d 00 31 00 31 00 66 00  7·4·6·a·-·1·1·f·
  00b0  31 00 2d 00 38 00 31 00 33 00 37 00 2d 00 66 00  1·-·8·1·3·7·-·f·
  00c0  37 00 36 00 37 00 30 00 35 00 62 00 37 00 62 00  7·6·7·0·5·b·7·b·
  00d0  62 00 65 00 61 00 15 98 74 79 

---

## media/mediaInfo.dat

Media manifest (131 bytes). References the `.spi` thumbnail with its **UTF-16LE filename**
and an **ASCII hex SHA-256 hash** for integrity verification. Ends with the `EOFX` marker.

| Offset | Size | Type | Content |
|--------|------|------|---------|
| `0x0E` | 2 | u16 | Filename string length (chars) |
| `0x10` | var | UTF-16LE | Media filename |
| after filename | 64 | ASCII | SHA-256 hash (hex-encoded) |
| last 4 | 4 | ASCII | `"EOFX"` marker |

In [ ]:
data = files["media/mediaInfo.dat"]
print(f"media/mediaInfo.dat — {len(data)} bytes\n")

if len(data) < 16:
    # No media attached to this note: just the 6-byte header + "EOFX" marker.
    print("No media referenced (manifest is just header + EOFX marker).")
else:
    str_len = struct.unpack_from("<H", data, 0x0E)[0]
    filename = data[0x10 : 0x10 + str_len * 2].decode("utf-16-le")
    hash_offset = 0x10 + str_len * 2
    sha256_stored = data[hash_offset : hash_offset + 64].decode("ascii")

    spi_key = f"media/{filename}"
    sha256_actual = hashlib.sha256(files[spi_key]).hexdigest()

    print(f"Filename:   media/{filename}")
    print(f"SHA-256:    {sha256_stored}")
    print(f"Verified:   {sha256_stored == sha256_actual}")
print(f"End marker: {data[-4:].decode('ascii')!r}")

In [42]:
print(hexdump(data))

  0000  18 15 00 00 00 00 45 4f 46 58                    ······EOFX


---

## note.note

Note-level metadata (1.7 KB). The most complex metadata file — it contains the **note title**
(UTF-16LE), **pen tool** class names (Java package paths like
`com.samsung.android.sdk.pen.pen.preload.FountainPen`), **layer UUIDs**, **page dimensions**,
and the **background color**.

The file uses a mix of fixed-header fields and variable-length TLV-style records.

In [43]:
data = files["note.note"]
print(f"note.note \u2014 {len(data)} bytes\n")

page_w = struct.unpack_from("<I", data, 0x28)[0]
page_h = struct.unpack_from("<I", data, 0x2C)[0]
print(f"Page dimensions: {page_w} \u00d7 {page_h}")

note.note — 1623 bytes

Page dimensions: 1600 × 4559


In [44]:
# Scan for ASCII strings >= 4 chars
print("--- ASCII strings ---")
s, start = "", 0
for i, b in enumerate(data):
    if 32 <= b < 127:
        if not s:
            start = i
        s += chr(b)
    else:
        if len(s) >= 4:
            print(f"  0x{start:04x}: {s!r}")
        s = ""
if len(s) >= 4:
    print(f"  0x{start:04x}: {s!r}")

# Scan for UTF-16LE strings >= 3 chars
print("\n--- UTF-16LE strings ---")
i = 0
while i < len(data) - 3:
    if 32 <= data[i] < 127 and data[i + 1] == 0:
        start = i
        chars = []
        while i < len(data) - 1 and 32 <= data[i] < 127 and data[i + 1] == 0:
            chars.append(chr(data[i]))
            i += 2
        text = "".join(chars)
        if len(text) >= 3:
            print(f"  0x{start:04x}: {text!r}")
    else:
        i += 1

--- ASCII strings ---
  0x0058: '42ab0f90-746a-11f1-ba30-67f01e837c6b'
  0x01bb: '42ab13e6-746a-11f1-a7c6-576487a359e0'
  0x05f4: 'A%%%'

--- UTF-16LE strings ---
  0x0131: 'OnlyPensBlacksize10'
  0x0345: '10;'
  0x034f: '0com.samsung.android.sdk.pen.pen.preload.BrushPen'
  0x03b7: '13;'
  0x03c1: '/com.samsung.android.sdk.pen.pen.preload.Pencil2'
  0x042f: '2com.samsung.android.sdk.pen.pen.preload.ObliquePen'
  0x049b: '18;0;100;'
  0x04b1: '3com.samsung.android.sdk.pen.pen.preload.FountainPen'
  0x0527: '/com.samsung.android.sdk.pen.pen.preload.InkPen2'
  0x058f: '0com.samsung.android.sdk.pen.pen.preload.BrushPen'


In [ ]:
bg = bg_color_from_note(data)
print(f"Background color: {bg}")

---

## SPI Thumbnail

The `.spi` file is stored uncompressed (`STORED` method in the ZIP), suggesting it's
already in a compressed format. Let's check if it matches any standard image format.

In [ ]:
spi_keys = [k for k in files if k.endswith(".spi")]

if not spi_keys:
    spi = None
    print("No .spi thumbnail in this archive (note has no media).")
else:
    spi_key = spi_keys[0]
    spi = files[spi_key]
    print(f"{spi_key} — {len(spi):,} bytes\n")

    magic_em_dash = "—"
    for fmt, check in {
        "PNG": spi[:4] == b"\x89PNG",
        "JPEG": spi[:2] == b"\xff\xd8",
        "BMP": spi[:2] == b"BM",
        "GIF": spi[:3] == b"GIF",
        "WEBP": spi[:4] == b"RIFF" and spi[8:12] == b"WEBP",
        "TIFF": spi[:2] in (b"II", b"MM"),
    }.items():
        print(f"  {fmt}: {'MATCH' if check else magic_em_dash}")

In [ ]:
if spi is None:
    print("(skipped — no .spi thumbnail)")
else:
    print("First 64 bytes (no known magic):")
    print(hexdump(spi, limit=64))
    print("\nVerdict: Samsung proprietary bitmap — not decodable with standard tools.")

---

## The `.page` File — Preview

The `.page` file contains all stroke geometry and attributes. At ~4.5 MB, it's by far the
largest file in the archive. Its detailed structure is covered in the next two notebooks.

In [48]:
page_key = [k for k in files if k.endswith(".page")][0]
page_data = files[page_key]
print(f"{page_key} \u2014 {len(page_data):,} bytes\n")

print("--- Header (first 128 bytes) ---")
print(hexdump(page_data, limit=128))

4302b93e-746a-11f1-9552-b7833a8a74d9.page — 25,376 bytes

--- Header (first 128 bytes) ---
  0000  ac 00 00 00 80 00 00 00 04 00 00 00 00 04 71 00  ··············q·
  0010  00 00 00 00 00 00 40 06 00 00 d6 08 00 00 00 00  ······@·········
  0020  00 00 00 00 00 00 24 00 34 00 33 00 30 00 32 00  ······$·4·3·0·2·
  0030  62 00 39 00 33 00 65 00 2d 00 37 00 34 00 36 00  b·9·3·e·-·7·4·6·
  0040  61 00 2d 00 31 00 31 00 66 00 31 00 2d 00 39 00  a·-·1·1·f·1·-·9·
  0050  35 00 35 00 32 00 2d 00 62 00 37 00 38 00 33 00  5·5·2·-·b·7·8·3·
  0060  33 00 61 00 38 00 61 00 37 00 34 00 64 00 39 00  3·a·8·a·7·4·d·9·
  0070  1b e4 46 aa 75 55 06 00 a0 0f 00 00 a0 0f 00 00  ··F·uU··········
  ... (25248 more bytes)


In [49]:
print("--- Footer (last 32 bytes) ---")
print(hexdump(page_data[-32:], offset=len(page_data) - 32))
print(f"\nFooter marker: {page_data[-26:].decode('ascii')!r}")

--- Footer (last 32 bytes) ---
  6300  53 cb cd 6a 32 51 50 61 67 65 20 66 6f 72 20 53  S··j2QPage for S
  6310  41 4d 53 55 4e 47 20 53 2d 50 65 6e 20 53 44 4b  AMSUNG S-Pen SDK

Footer marker: 'Page for SAMSUNG S-Pen SDK'


---

## Summary

| File | Size | Purpose |
|------|------|---------|
| `end_tag.bin` | 144 B | Document footer — created/modified timestamps (i64 ms epoch) |
| `pageIdInfo.dat` | 140 B | Page UUID registry with 32-byte integrity hashes |
| `media/mediaInfo.dat` | 131 B | Media manifest — filename (UTF-16LE) + SHA-256 verification |
| `note.note` | ~1.7 KB | Note title, pen tool names, background color, layer UUIDs |
| `media/*.spi` | ~454 KB | Page thumbnail — Samsung proprietary format, not decodable |
| `<uuid>.page` | ~4.5 MB | Stroke geometry and attributes — see notebooks 02 and 03 |

Each file has a recognizable footer marker:
- `end_tag.bin` → `"Document for S-Pen SDK"`
- `<uuid>.page` → `"Page for SAMSUNG S-Pen SDK"`
- `mediaInfo.dat` → `"EOFX"`

**Next:** [02_strokes.ipynb](02_strokes.ipynb) dives into the `.page` file to decode stroke geometry.